# 02 — Build and inspect the GRPO dataset

SFT teaches the desired response pattern. GRPO then samples several answers to the
same task and increases the probability of answers that receive higher reward.

The student-facing prompt is still only a self-contained mechanism task. Hidden
reference fields are passed to reward functions, not appended to the prompt.

## Configuration

The OpenAI key is read from `OPENAI_API_KEY`. All other settings are explicit
notebook variables. Hugging Face uses your cached `hf auth login` credentials
unless you deliberately uncomment the optional token line.

In [ ]:
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import JSON, Markdown, display

from science_course.data import (
    build_grpo_dataset,
    read_jsonl,
    render_completion,
)
from science_course.hub import require_hf_namespace
from science_course.judge import MechanismJudge
from science_course.rewards import component_scores
from science_course.teacher import DEFAULT_TEACHER_MODEL, require_openai_key

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
CANONICAL = DATA / "canonical" / "mechanism_tasks.jsonl"
GRPO_DISK = DATA / "processed" / "grpo"
JUDGE_CACHE = ROOT / "results" / "grpo_judge_cache.jsonl"

JUDGE_MODEL = DEFAULT_TEACHER_MODEL
DATASET_HF_REPO = "lamm-mit/scientific-sft-grpo-data"
PUSH_DATASETS_TO_HUB = True
DATASET_PRIVATE = False
HF_TOKEN = None
# HF_TOKEN = os.environ["HF_TOKEN"]  # Optional; prefer `hf auth login`.

require_openai_key()
assert JUDGE_MODEL == "gpt-5.6-terra", (
    "This class notebook is tested with the requested judge model: "
    "gpt-5.6-terra"
)
if PUSH_DATASETS_TO_HUB:
    require_hf_namespace(DATASET_HF_REPO, token=HF_TOKEN)
canonical = read_jsonl(CANONICAL)
if not canonical:
    raise RuntimeError("Run notebook 01 first.")
sns.set_theme(style="whitegrid", context="talk")
print(
    {
        "canonical_tasks": len(canonical),
        "judge_model": JUDGE_MODEL,
        "dataset_hub_repo": DATASET_HF_REPO,
    }
)

## 1. Make the GRPO projection

Each row contains:

- `prompt`: the only input to the policy;
- `task`: used to verify that quoted evidence actually appears in the task;
- `mechanism_steps` and `causal_links`: a hidden semantic rubric;
- reference reasoning/answer: guidance for the judge, not a phrase-match target;
- provenance fields.

In [ ]:
grpo = build_grpo_dataset(canonical)
if len(grpo["train"]) == 0 or len(grpo["validation"]) == 0:
    raise RuntimeError(
        "The paper-level split produced an empty GRPO partition. Increase "
        "MAX_SOURCE_PAPERS in notebook 01, then rerun generation."
    )
if GRPO_DISK.exists():
    shutil.rmtree(GRPO_DISK)
grpo.save_to_disk(GRPO_DISK)
for split_name, split_data in grpo.items():
    split_data.to_parquet(DATA / "processed" / f"grpo_{split_name}.parquet")
if PUSH_DATASETS_TO_HUB:
    grpo.push_to_hub(
        DATASET_HF_REPO,
        config_name="grpo",
        private=DATASET_PRIVATE,
        token=HF_TOKEN,
        commit_message="Publish mechanism GRPO splits",
    )

for split_data in grpo.values():
    assert "source_text" not in split_data.column_names
    assert "completion" not in split_data.column_names

print(grpo)

In [ ]:
row = grpo["train"][0]
display(Markdown("### What the policy sees"))
display(JSON(row["prompt"]))
display(Markdown("### What reward functions can see"))
display(
    JSON(
        {
            key: row[key]
            for key in (
                "task",
                "mechanism_steps",
                "causal_links",
                "required_concepts",
                "reference_answer",
            )
        }
    )
)

## 2. Reward design: mechanics plus semantics

Four complementary signals are used:

1. **Structure** — the three required response tags are present.
2. **Evidence grounding** — the quoted evidence occurs in the task.
3. **Concept coverage** — expected concepts appear somewhere in the explanation.
4. **Mechanism judge** — `gpt-5.6-terra` evaluates causal correctness,
   completeness, and whether evidence supports the stated mechanism.

The semantic judge receives a whole trainer batch in one structured-output API
request. Judgments are cached by content hash, including model and prompt version.
A keyword check cannot tell causal direction; the judge carries most of the reward.

In [ ]:
reference_completion = render_completion(
    {
        "reasoning": row["reference_reasoning"],
        "evidence": row["reference_evidence"],
        "answer": row["reference_answer"],
    }
)
deterministic = component_scores(
    reference_completion,
    task=row["task"],
    reference_answer=row["reference_answer"],
    required_concepts=row["required_concepts"],
)
display(JSON(deterministic))

In [ ]:
judge = MechanismJudge(
    model=JUDGE_MODEL,
    cache_path=JUDGE_CACHE,
)
semantic_score = judge.score(
    [
        {
            "task": row["task"],
            "reference_reasoning": row["reference_reasoning"],
            "reference_answer": row["reference_answer"],
            "mechanism_steps": row["mechanism_steps"],
            "causal_links": row["causal_links"],
            "student_response": reference_completion,
        }
    ]
)[0]
print({"cached_semantic_mechanism_score": semantic_score})

## 3. Inspect the mechanism curriculum

The purpose of this audit is diversity of causal structure, not numerical label
balance. We inspect task length, number of causal links, and concept vocabulary.

In [ ]:
frame = pd.DataFrame(
    [
        {
            "split": split_name,
            "task_chars": len(item["task"]),
            "causal_links": len(item["causal_links"]),
            "mechanism_steps": len(item["mechanism_steps"]),
        }
        for split_name, split_data in grpo.items()
        for item in split_data
    ]
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.boxplot(frame, x="split", y="task_chars", ax=axes[0], color="#87a9cc")
axes[0].tick_params(axis="x", rotation=25)
axes[0].set_title("Self-contained task size")
sns.countplot(
    frame,
    x="causal_links",
    hue="split",
    ax=axes[1],
    palette="deep",
)
axes[1].set_title("Explicit causal links")
plt.tight_layout()
plt.show()

In [ ]:
concept_counts = (
    pd.Series(
        [
            concept.casefold()
            for split_data in grpo.values()
            for item in split_data
            for concept in item["required_concepts"]
        ]
    )
    .value_counts()
    .head(20)
    .sort_values()
)
concept_counts.plot.barh(
    figsize=(9, 7),
    color="#d97732",
    title="Frequent rubric concepts",
)
plt.xlabel("tasks")
plt.tight_layout()
plt.show()

## Result

The GRPO policy will receive no answer and no source document—only a
self-contained mechanism task. The hidden causal rubric supports semantic reward.

Continue with **03_finetune_sft_lora.ipynb**.